# 06. 결과 분석 및 시각화

실험 결과 종합 분석 및 최종 보고서 작성

## 1. 환경 설정

In [ ]:
import sys
sys.path.append('..')

import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.visualization import (
    plot_chunk_distribution,
    plot_latency_comparison,
    create_comparison_table
)

# 스타일 설정
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

## 2. 데이터 로드

In [ ]:
# 모든 결과 로드
strategies = ['fixed', 'recursive', 'semantic']
all_chunks = {}
all_stats = {}
all_results = {}

for strategy in strategies:
    base_path = Path(f'../results/{strategy}_chunking')
    
    try:
        with open(base_path / 'chunks.json', 'r') as f:
            all_chunks[strategy] = json.load(f)
        with open(base_path / 'stats.json', 'r') as f:
            all_stats[strategy] = json.load(f)
        with open(base_path / 'retrieval_results.json', 'r') as f:
            all_results[strategy] = json.load(f)
        print(f"{strategy}: {len(all_chunks[strategy])} chunks 로드")
    except FileNotFoundError:
        print(f"{strategy}: 파일 없음")

## 3. 종합 비교 테이블

In [ ]:
# 비교 데이터프레임 생성
comparison_data = []

for strategy in strategies:
    if strategy not in all_stats:
        continue
    
    stats = all_stats[strategy]
    comparison_data.append({
        'Strategy': strategy.capitalize(),
        'Total Chunks': stats.get('total_chunks', 0),
        'Avg Size (words)': round(stats.get('avg_size', 0), 1),
        'Std Dev': round(stats.get('std_size', 0), 1),
        'Min Size': stats.get('min_size', 0),
        'Max Size': stats.get('max_size', 0),
        'Uniformity': round(stats.get('size_uniformity', 0), 3),
        'Time (s)': round(stats.get('chunking_time', 0), 2)
    })

df_comparison = pd.DataFrame(comparison_data)
print("\n=== 청킹 전략 종합 비교 ===")
print(df_comparison.to_string(index=False))

## 4. 청크 크기 분포 분석

In [ ]:
# 청크 크기 분포 시각화
chunk_sizes = {}

for strategy in strategies:
    if strategy in all_chunks:
        sizes = [c.get('word_count', len(c.get('text', '').split())) for c in all_chunks[strategy]]
        chunk_sizes[strategy] = sizes

if chunk_sizes:
    fig = plot_chunk_distribution(
        chunk_sizes,
        save_path='../results/comparison/chunk_size_distribution_final.png'
    )
    plt.show()

In [ ]:
# 박스플롯 비교
fig, ax = plt.subplots(figsize=(10, 6))

box_data = [chunk_sizes.get(s, []) for s in strategies if s in chunk_sizes]
labels = [s.capitalize() for s in strategies if s in chunk_sizes]

bp = ax.boxplot(box_data, labels=labels, patch_artist=True)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('Chunk Size (words)')
ax.set_title('Chunk Size Distribution by Strategy')
plt.tight_layout()
plt.savefig('../results/comparison/chunk_size_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. 처리 시간 분석

In [ ]:
# 처리 시간 비교
latencies = {s: all_stats.get(s, {}).get('chunking_time', 0) for s in strategies if s in all_stats}

if latencies:
    fig = plot_latency_comparison(
        latencies,
        save_path='../results/comparison/latency_comparison_final.png'
    )
    plt.show()

## 6. 효율성 분석

In [ ]:
# 청크 생성 효율성 (chunks/second)
efficiency_data = []

for strategy in strategies:
    if strategy not in all_stats:
        continue
    
    stats = all_stats[strategy]
    total_chunks = stats.get('total_chunks', 0)
    time_taken = stats.get('chunking_time', 1)
    
    efficiency_data.append({
        'Strategy': strategy.capitalize(),
        'Chunks': total_chunks,
        'Time (s)': round(time_taken, 2),
        'Chunks/sec': round(total_chunks / max(time_taken, 0.001), 1)
    })

df_efficiency = pd.DataFrame(efficiency_data)
print("\n=== 처리 효율성 ===")
print(df_efficiency.to_string(index=False))

## 7. 샘플 청크 품질 비교

In [ ]:
# 각 전략의 첫 번째 청크 비교
print("\n=== 샘플 청크 비교 ===")

for strategy in strategies:
    if strategy not in all_chunks or not all_chunks[strategy]:
        continue
    
    chunk = all_chunks[strategy][0]
    text = chunk.get('text', '')
    word_count = chunk.get('word_count', len(text.split()))
    
    print(f"\n[{strategy.upper()}] ({word_count} words)")
    print("-" * 50)
    print(text[:300] + "..." if len(text) > 300 else text)

## 8. 결론 및 권장사항

In [ ]:
# 결론 도출
print("\n" + "=" * 60)
print("실험 결론")
print("=" * 60)

if all_stats:
    # 가장 균일한 청크 크기
    uniformity_scores = {s: all_stats[s].get('size_uniformity', 0) for s in strategies if s in all_stats}
    most_uniform = max(uniformity_scores, key=uniformity_scores.get)
    
    # 가장 빠른 처리
    time_scores = {s: all_stats[s].get('chunking_time', float('inf')) for s in strategies if s in all_stats}
    fastest = min(time_scores, key=time_scores.get)
    
    print(f"\n1. 청크 크기 균일성")
    print(f"   - 가장 균일: {most_uniform.capitalize()} (uniformity: {uniformity_scores[most_uniform]:.3f})")
    print(f"   - Fixed는 고정 크기로 가장 균일하지만 문맥이 끊길 수 있음")
    
    print(f"\n2. 처리 속도")
    print(f"   - 가장 빠름: {fastest.capitalize()} ({time_scores[fastest]:.2f}초)")
    print(f"   - Semantic은 임베딩이 필요하여 가장 느림")
    
    print(f"\n3. 권장사항")
    print(f"   - 농업 문서(매뉴얼, 가이드): Recursive Chunking 권장")
    print(f"     (문서 구조 보존, 적절한 속도)")
    print(f"   - 실시간 처리 필요: Fixed Chunking 권장")
    print(f"     (가장 빠름, 구현 간단)")
    print(f"   - 최고 품질 필요: Semantic Chunking 권장")
    print(f"     (의미 보존, 오프라인 처리 시 적합)")

## 9. 최종 보고서 생성

In [ ]:
# 마크다운 보고서 생성
report = f"""# RAG 청킹 전략 비교 실험 결과

## 실험 개요

- **목적**: 농업 도메인 문서에 최적화된 청킹 전략 도출
- **비교 전략**: Fixed-size, Recursive, Semantic
- **문서 수**: {len(all_chunks.get('fixed', all_chunks.get('recursive', [])))} documents

## 청킹 전략 비교

{df_comparison.to_markdown(index=False)}

## 주요 발견

### 1. Fixed-size Chunking
- **장점**: 가장 빠른 처리 속도, 균일한 청크 크기
- **단점**: 문맥 단절 가능성, 문장 중간에서 분할 발생
- **적합**: 실시간 처리, 대용량 데이터

### 2. Recursive Chunking
- **장점**: 문서 구조 보존, 자연스러운 경계에서 분할
- **단점**: 청크 크기 불균등
- **적합**: 매뉴얼, 가이드 등 구조화된 문서

### 3. Semantic Chunking
- **장점**: 의미 단위 보존, 주제별 일관성
- **단점**: 느린 처리 속도, 임베딩 모델 의존
- **적합**: 고품질 요구, 오프라인 인덱싱

## 결론

**농업 도메인 문서에는 Recursive Chunking을 권장합니다.**

이유:
1. 매뉴얼, 가이드 등 농업 문서는 섹션 구조가 명확함
2. 문서 구조를 보존하여 검색 품질 향상
3. Fixed보다 정확하고, Semantic보다 빠름

---
*생성일: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}*
"""

# 보고서 저장
with open('../results/comparison/final_report.md', 'w', encoding='utf-8') as f:
    f.write(report)

print("최종 보고서 저장 완료: results/comparison/final_report.md")

## 10. 실험 완료

모든 실험이 완료되었습니다!

### 생성된 결과물

```
results/
├── fixed_chunking/
│   ├── chunks.json
│   ├── retrieval_results.json
│   └── stats.json
├── recursive_chunking/
│   ├── chunks.json
│   ├── retrieval_results.json
│   └── stats.json
├── semantic_chunking/
│   ├── chunks.json
│   ├── retrieval_results.json
│   └── stats.json
└── comparison/
    ├── chunk_size_distribution.png
    ├── latency_comparison.png
    ├── evaluation_results.json
    ├── chunk_comparison.csv
    └── final_report.md
```